In [2]:
import pandas as pd
import re

# =========================
# FILE PATHS
# =========================
file_path = "C:/Users/Ex0164/Book1.xlsx"
unique_file = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"

# =========================
# HZ SHEET LOGIC
# =========================
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns in HZ
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines_hz = pd.unique(hz[machine_cols].values.ravel())
machines_hz = [m for m in machines_hz if pd.notna(m)]

# Extract tonnage from machine names
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', str(machine))
    if match:
        return int(match.group(1))
    return None

machine_tonnage_hz = {m: extract_machine_tonnage(m) for m in machines_hz}

# Parse comma-separated tonnage values in HZ
def parse_tonnage(x):
    if pd.isna(x):
        return []
    tonnages = []
    for t in str(x).split(","):
        t = t.strip()
        match = re.search(r'(\d+)', t)
        if match:
            tonnages.append(int(match.group(1)))
    return tonnages

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# Build HZ compatibility matrix
hz_matrix_data = []
for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]
    row_data = {"Part": part}
    for machine in machines_hz:
        m_ton = machine_tonnage_hz[machine]
        row_data[machine] = 1 if m_ton in tonnage_list else 0
    hz_matrix_data.append(row_data)

hz_matrix = pd.DataFrame(hz_matrix_data)

# =========================
# VT SHEET LOGIC
# =========================
vt = pd.read_excel(file_path, sheet_name="VT")
unique = pd.read_excel(unique_file, sheet_name="Sheet1")

# Clean columns
vt.columns = vt.columns.str.strip()
unique.columns = unique.columns.str.strip()

# Clean VT data
vt = vt.dropna(subset=["Part", "Tons"]).copy()
vt["Part"] = vt["Part"].astype(str).str.strip()

# Extract part tonnage list from comma-separated values
def extract_part_tons(text):
    numbers = re.findall(r'\d+', str(text))
    return list(set(int(n) for n in numbers))

vt["Tons_List"] = vt["Tons"].apply(extract_part_tons)

# Clean machine data
unique = unique.dropna(subset=["Unique Machines", "Tonnage"]).copy()
unique["Unique Machines"] = unique["Unique Machines"].astype(str).str.strip()
unique["Machine_Ton"] = unique["Tonnage"].apply(lambda x: int(re.search(r'\d+', str(x)).group()))

# Machine → tonnage mapping
machine_tonnage_vt = dict(zip(unique["Unique Machines"], unique["Machine_Ton"]))
machines_vt = list(machine_tonnage_vt.keys())

# Build VT compatibility matrix
vt_matrix_data = []
for _, row in vt.iterrows():
    part = row["Part"]
    part_tons = row["Tons_List"]
    row_data = {"Part": part}
    for machine in machines_vt:
        machine_ton = machine_tonnage_vt[machine]
        row_data[machine] = 1 if machine_ton in part_tons else 0
    vt_matrix_data.append(row_data)

vt_matrix = pd.DataFrame(vt_matrix_data)

# =========================
# SAVE BOTH MATRICES
# =========================
with pd.ExcelWriter(output_path, engine="openpyxl", mode="w") as writer:
    hz_matrix.to_excel(writer, sheet_name="HZ_Matrix", index=False)
    vt_matrix.to_excel(writer, sheet_name="VT_Matrix", index=False)

print("✅ Compatibility matrices saved to HZ_Matrix and VT_Matrix sheets")


✅ Compatibility matrices saved to HZ_Matrix and VT_Matrix sheets
